# Phoundry — SOC Blue-Team Email Triage

Analyst-triggered triage of recent email, using **Claude on Microsoft Foundry** via the
**Microsoft Agent Framework**, enriched with **Sublime Security** and **VirusTotal**.

Run the cells in order. Nothing happens on a schedule — you are the trigger.

---

### Before you run

| | |
|---|---|
| **Azure sign-in** | `az login` — the agent authenticates as *you*, so Foundry and Sublime audit trails name a person, not a shared secret. |
| **Config** | Copy `.env.example` to `.env` and fill it in. |
| **Install** | `pip install -e ".[notebook,msticpy]"` |

### Two things to know

1. **This notebook renders live phishing content.** Indicators are defanged (`hxxp://`, `[.]`)
   and all rendering is HTML-escaped, but the subject lines and body text are real attacker
   output. Treat the outputs accordingly.
2. **Saved notebooks embed their outputs.** A committed `.ipynb` will contain recipient
   addresses, message bodies and IOCs. Clear outputs before sharing:
   ```
   jupyter nbconvert --clear-output --inplace notebooks/*.ipynb
   ```
   `.gitignore` already excludes `reports/` and `.env`.

## 1 · Connect

Loads config and opens clients. The summary below is redacted — safe to screenshot.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
from IPython.display import HTML, Markdown, display

from soc_triage.config import load_settings
from soc_triage.report import render_html, render_markdown, render_queue, save_report
from soc_triage.triage import TriageSession

settings = load_settings()
session = TriageSession(settings)

display(HTML(
    '<table style="font-family:-apple-system,sans-serif;font-size:13px">'
    + ''.join(f'<tr><td style="color:#777;padding:2px 12px 2px 0">{k}</td>'
              f'<td><code>{v}</code></td></tr>'
              for k, v in settings.describe().items())
    + '</table>'
))

## 2 · Pull recent messages

Queries Sublime for messages ingested in the lookback window
(`GET /v0/message-groups/search` with an inclusive `created_at[gte]` and exclusive `created_at[lt]`).

**The 5-minute default matches the original monitoring requirement, and in a quiet tenant it will
often return nothing.** That is correct behavior, not a bug — widen `LOOKBACK_MINUTES` to demo.

In [ ]:
LOOKBACK_MINUTES = 5      # widen to 60 / 1440 in a low-volume test tenant
INBOUND_ONLY = True       # type=inbound — exclude your own outbound and internal mail

messages = session.recent_messages(lookback_minutes=LOOKBACK_MINUTES, inbound_only=INBOUND_ONLY)

if not messages:
    display(Markdown(
        f'**No messages in the last {LOOKBACK_MINUTES} minutes.** '
        'Increase `LOOKBACK_MINUTES` and re-run this cell.'
    ))
else:
    df = pd.DataFrame(messages)[[
        'message_id', 'created_at', 'sender', 'subject', 'mailbox',
        'flagged_rules', 'group_size', 'user_reports',
    ]]
    df['flagged_rules'] = df['flagged_rules'].apply(lambda r: ', '.join(r) if r else '—')
    print(f'{len(messages)} message(s) in the last {LOOKBACK_MINUTES} minutes\n')
    display(df.style.hide(axis='index'))

## 2b · Look up a specific person

The analyst-driven entry point: type an email address or a display name instead of waiting for
a time window to produce something.

`search_message_groups` filters on `created_at` only, so an identifier has to go through a
**hunt** (`POST /v0/hunt-jobs`) with generated MQL. Hunt jobs are asynchronous — this cell blocks
for a few seconds while the job runs.

Results land in `messages`, so **section 3 triages them without any change**. Leave `IDENTIFIER`
empty to skip this cell and keep the time-window results from section 2.


In [ ]:
IDENTIFIER = ''       # 'alice@corp.com' (exact address) or 'Jane Doe' (display-name substring)
LOOKBACK_DAYS = 7     # hunts run over a range, not a live window

if not IDENTIFIER:
    display(Markdown('_Skipped — set `IDENTIFIER` to look up a specific person._'))
else:
    from soc_triage.sublime import build_identity_mql

    # Print the query rather than hiding it. The analyst should be able to see
    # exactly what was asked of Sublime before trusting what comes back.
    print(f'MQL: {build_identity_mql(IDENTIFIER)}\n')

    found = session.find_messages(IDENTIFIER, days=LOOKBACK_DAYS)

    if not found:
        display(Markdown(
            f'**No messages involving `{IDENTIFIER}` in the last {LOOKBACK_DAYS} day(s).** '
            'Widen `LOOKBACK_DAYS`, or check the identifier — a display name is matched as a '
            'case-insensitive substring, but an address must match exactly.'
        ))
    else:
        messages = found          # section 3 triages whatever is in `messages`
        df = pd.DataFrame(messages)[[
            'message_id', 'created_at', 'sender', 'subject', 'mailbox',
            'flagged_rules', 'group_size', 'user_reports',
        ]]
        df['flagged_rules'] = df['flagged_rules'].apply(lambda r: ', '.join(r) if r else '—')
        print(f'{len(messages)} message(s) involving {IDENTIFIER} in the last {LOOKBACK_DAYS} day(s)\n')
        display(df.style.hide(axis='index'))


## 3 · Triage

Each message gets an independent agent run. The agent reads the message, computes
authentication results deterministically, extracts and prioritizes indicators, enriches
selectively against its VirusTotal budget, compares itself to Sublime's own verdict, and
scopes the campaign if there is one.

Anything malicious, low-confidence, disagreeing with Sublime, or containing a prompt-injection
attempt is automatically re-run on the escalation model for an independent second opinion.

In [ ]:
# Triage everything from the window, or paste specific ids here.
TARGET_IDS = [m['message_id'] for m in messages][:10]

# Bounded concurrency — pay-as-you-go Foundry quota for Claude is 40 RPM / 40K ITPM,
# and each triage makes several tool-augmented model calls.
results = await session.triage_many(TARGET_IDS, concurrency=3)

display(HTML(render_queue(results)))

## 4 · Read the reports

Worst first. Watch for the banners — they mark the things a human should look at before
anything else:

| Banner | Meaning |
|---|---|
| 🚨 **Confirmed interaction** | Someone already clicked. This is an incident, not a triage item. |
| 💉 **Prompt injection** | The email tried to manipulate the analyzing model. Reported, not obeyed. |
| 🚩 **Disagrees with Sublime** | The agent and the platform reached different conclusions. Highest-value signal in the report. |

In [ ]:
ordered = sorted(results, key=lambda r: (-r.verdict.severity.rank, -r.verdict.confidence))

for result in ordered:
    display(HTML(render_html(result)))
    if result.error:
        display(Markdown(f'> ⚠️ Triage error: `{result.error}`'))

In [ ]:
# Save Markdown reports for ticketing.
for result in results:
    if result.ok:
        path = save_report(result, settings.report_dir)
        print(f'{result.verdict.severity.value:<11} → {path.name}')

## 5 · Response actions — human in the loop

The agent has **no mailbox authority** unless `ALLOW_MAILBOX_ACTIONS=true` in `.env`.
By default it recommends; you decide and execute.

This cell shows what the agent recommended and executes only what you uncomment. Actions run
under *your* Sublime API key, so the audit trail attributes them to you.

In [ ]:
actionable = [r for r in results if r.ok and r.verdict.is_actionable]

if not actionable:
    display(Markdown('No messages met the threshold for a response action.'))
else:
    for r in actionable:
        v = r.verdict
        print(f'{v.severity.value.upper():<10} {v.recommended_disposition.value:<16} '
              f'{v.confidence:.0%}  {v.subject[:55]}')
        print(f'{"":10} session.sublime.action_message("{v.message_id}", "quarantine")')
        print()

# Uncomment a specific line above and run it to execute that action.
# There is deliberately no 'apply all' — each action is an individual decision.

## 6 · Triage an arbitrary message

For a user-reported phish or anything outside the window. Paste a Sublime message ID.

In [ ]:
MESSAGE_ID = ''  # e.g. from the Sublime dashboard URL

if MESSAGE_ID:
    result = await session.triage(
        MESSAGE_ID,
        justification='Analyst-initiated review of user-reported message',
    )
    display(HTML(render_html(result)))
    display(Markdown('---\n### Tool calls\n'
                     + '\n'.join(f'{i}. `{c}`' for i, c in enumerate(result.tool_calls, 1))))
else:
    display(Markdown('_Set `MESSAGE_ID` above to triage a specific message._'))

## 7 · Close

Releases HTTP clients. Safe to skip — closing the kernel does the same thing.

In [ ]:
session.close()
print('Session closed.')